In [ ]:
"""
Premier League - Open-Play (non-set-piece) Cross Models
=========================================================
Trains 6 XGBoost models on every non-set-piece cross in the Premier League
2025-26 event data (380 matches, Opta/Stats Perform feed format):

  1. cross_completion          - P(cross is retained by a teammate)
  2. cross_chance_creation     - P(cross directly leads to a shot)
  3. cross_goal_contribution   - P(cross directly leads to a goal)   ("xCrossGoal")
  4. cross_defended            - P(defending team immediately clears/intercepts it)
  5. cross_delivery_value      - continuous danger value of the cross (regression)
  6. cross_outcome_multiclass  - {incomplete, complete_no_shot, shot_no_goal, goal}

Also trains a 7th model chained onto cross_defended: a headed-clearance landing
model (same methodology as Ecuador 2026/Scripts/header_clearance_landing_model.py
-- MultiOutputRegressor(GradientBoostingRegressor) predicting landing x/y from
every typeId-12/qualifier-15 headed clearance in these same match files). It is
applied to the crosses that were cleared by a header, so each of those also gets
a predicted clearance-landing zone, plus a per-team "clears furthest" leaderboard.
Outputs mirror the Ecuador folder layout under Premier League/ClearanceLandingModel/.

A cross is identified as pass event with qualifierId 2 ("Cross"). It is treated
as open play (non-set-piece) when none of the following qualifiers are present:
free kick taken (5), corner taken (6), set piece (24), from corner (25),
direct free kick (26), throw-in set piece (160). Shots are linked back to the
cross that created them via qualifierId 55 (RELATED_EVENT_ID).

Note on the "expected" columns: PASS_END_X/Y (qualifiers 140/141) are the actual
end location of the pass attempt (not a hand-labelled "intended target"), so for
incomplete crosses this is where the ball was won/went out, not necessarily where
the crosser aimed. This is standard practice for pass-outcome models built on
event data (public xPass-style models use it the same way) but it does mean
"expected completion" partly reflects where the cross ended up, not just its
difficulty at the moment of delivery. Treat the completion model as a target-zone-
conditioned difficulty baseline, not a pre-strike intent model.

Also saves two families of visuals to Cross Models/visuals/:

1. Presentation/media pieces in the Waltzing Analytics dark brand style
   (matches Ecuador 2026/Scripts/key_passes_into_box.py):
     - visuals/cross_maps/{team}.png -- per-team pitch map of every open-play
       cross that created a shot this season (green = goal, amber = shot only)
     - visuals/rest_attack_maps/{team}.png -- per-team second-ball recovery map
     - 03_cross_origin_heatmaps.png, 04_top_crossers.png,
       05_team_attack_vs_defend.png, 06_delivery_type_breakdown.png,
       07_clearance_landing.png, 08_second_ball_recovery_zones.png,
       09_second_ball_recovery_by_side.png

2. Methods-section diagnostics in a classic scientific-paper style (white
   background, Okabe-Ito colorblind-safe palette, serif type, figure
   numbers/captions with explicit methodology notes):
     - 01_model_performance_summary.png -- AUC ± SD, regression/multiclass
       fold-score boxplots
     - 02_roc_curves.png -- per-fold ROC curves with mean ± SD band
     - 10_calibration_curves.png -- reliability diagrams (reveals that
       completion/chance-creation are over-confident due to scale_pos_weight)
     - 11_feature_importance.png -- top-8 XGBoost gain per model
     - 12_confusion_matrix.png -- reveals the 4-class model rarely predicts
       the rare classes (shot_no_goal, goal) despite 79.7% raw accuracy
     - 13_residual_diagnostics.png -- predicted vs. actual + residual
       distribution for the delivery-value regression
"""
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from mplsoccer import Pitch, VerticalPitch
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, brier_score_loss, confusion_matrix, f1_score, log_loss,
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------
# Chart style -- Waltzing Analytics dark brand style, matching
# Ecuador 2026/Scripts/key_passes_into_box.py so all WA output looks consistent.
# ----------------------------------------------------------------------------
C_NAVY, C_INDIGO, C_PURPLE, C_PINK = '#2f8fd1', '#7b7fd6', '#c179d1', '#f06fa3'
C_CORAL, C_AMBER, C_GREEN = '#ff8a75', '#ffc247', '#4ade80'
BG, PITCH_LINE = '#0d1117', '#2c3a4d'
TXT_SUB, TXT_CAP, TXT_MUTE = '#9aa4b2', '#c7ccd4', '#6b7684'
LOGO_PATH = '/Users/marclamberts/Downloads/Waltzing Analytics Logo Type.png'

plt.rcParams['font.family'] = 'DejaVu Sans'


def add_logo(fig, width=0.15, margin=0.016):
    import matplotlib.image as mpimg
    try:
        img = mpimg.imread(LOGO_PATH)
    except FileNotFoundError:
        return
    fig_w, fig_h = fig.get_size_inches()
    img_h, img_w = img.shape[0], img.shape[1]
    width_in = width * fig_w
    height_in = width_in * (img_h / img_w)
    height = height_in / fig_h
    left = 1 - margin - width
    bottom = 1 - margin - height
    logo_ax = fig.add_axes([left, bottom, width, height], zorder=10)
    logo_ax.patch.set_alpha(0)
    logo_ax.set_xlim(0, img_w)
    logo_ax.set_ylim(img_h, 0)
    logo_ax.imshow(img)
    logo_ax.axis('off')


def add_footer(fig, note):
    # scale down for narrower figures so the note never runs into the signature
    fig_w = fig.get_size_inches()[0]
    scale = min(1.0, fig_w / 11.0)
    fig.text(0.98, 0.006, 'Marc Lamberts', fontsize=9.5 * scale, ha='right', color=TXT_MUTE, style='italic')
    fig.text(0.02, 0.006, note, fontsize=7.3 * scale, color=TXT_MUTE, ha='left')


def style_axes_dark(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=TXT_MUTE, labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(PITCH_LINE)
    ax.xaxis.label.set_color(TXT_CAP)
    ax.yaxis.label.set_color(TXT_CAP)
    ax.grid(color=PITCH_LINE, lw=0.6, alpha=0.6)


FOOTER_NOTE = ('Data via Opta | Premier League 2025-26 event data · open-play crosses only '
               '(excludes corners/free kicks/throw-ins) · shot linked via qualifier 55')

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
SOURCE_DIR = Path('/Users/marclamberts/Event data/Premier League')
OUTPUT_DIR = Path('/Users/marclamberts/Event data/Premier League/Cross Models')
MODEL_DIR = OUTPUT_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
RANDOM_STATE = 42
MIN_CROSSES_FOR_LEADERBOARD = 15

# ----------------------------------------------------------------------------
# Opta / Stats Perform qualifier IDs (cross-checked against
# xg_output/model_meta qualifier_mapping and validated on sample match files).
# ----------------------------------------------------------------------------
Q = {
    'CROSS': 2, 'FREEKICK_TAKEN': 5, 'CORNER_TAKEN': 6,
    'HEADER': 15, 'RIGHT_FOOT': 20, 'OTHER_BODY': 21,
    'REGULAR_PLAY': 22, 'FAST_BREAK': 23, 'SET_PIECE': 24, 'FROM_CORNER': 25, 'DIRECT_FREE_KICK': 26,
    'ASSISTED': 29, 'RELATED_EVENT_ID': 55, 'LEFT_FOOT': 72, 'BLOCKED': 82,
    'INTENTIONAL_ASSIST': 154, 'PULL_BACK': 195, 'THROWIN_SETPIECE': 160, 'BIG_CHANCE': 214,
    'PASS_END_X': 140, 'PASS_END_Y': 141, 'LENGTH': 212, 'ANGLE': 213, 'DIRECTION': 56,
}
CLEARANCE_TYPE_ID = 12
SET_PIECE_QUALIFIERS = {Q['FREEKICK_TAKEN'], Q['CORNER_TAKEN'], Q['SET_PIECE'],
                         Q['FROM_CORNER'], Q['DIRECT_FREE_KICK'], Q['THROWIN_SETPIECE']}
SHOT_TYPE_IDS = {13, 14, 15, 16}          # Attempt Saved, Post, Miss, Goal
GOAL_TYPE_ID = 16
PASS_TYPE_ID = 1
ON_BALL_TYPE_IDS = {1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 44, 45, 49, 50, 51, 60, 68}
DEFENSIVE_STOP_TYPE_IDS = {8, 12}         # Interception, Clearance

NUM_FEATURES = [
    'x', 'y', 'end_x', 'end_y', 'length', 'angle', 'minute',
    'dist_to_goal_start', 'dist_to_goal_end', 'byline_proximity',
    'delivery_depth', 'lateral_shift', 'end_zone_y_dist_center', 'pull_back',
]
CAT_FEATURES = ['body_part', 'phase', 'wide_channel', 'period']
FEATURE_COLS = NUM_FEATURES + CAT_FEATURES


# ----------------------------------------------------------------------------
# Step 1: extract open-play (non-set-piece) crosses from every match file
# ----------------------------------------------------------------------------
def qmap(event):
    return {q['qualifierId']: q.get('value') for q in event.get('qualifier', [])}


def parse_match_file(path: Path):
    with open(path) as f:
        data = json.load(f)
    events = [e for e in data['event'] if e.get('periodId') in (1, 2)]
    events.sort(key=lambda e: e['timeStamp'])
    return data['matchDetails'], events


def extract_crosses(path: Path):
    match_name = path.stem
    try:
        home_team, away_team = match_name.split('_', 1)[1].split(' - ')
    except Exception:
        home_team, away_team = None, None

    _, events = parse_match_file(path)

    shot_by_related = {}
    for s in events:
        if s['typeId'] in SHOT_TYPE_IDS:
            rel = qmap(s).get(Q['RELATED_EVENT_ID'])
            if rel is not None:
                shot_by_related.setdefault(int(rel), s)

    on_ball = [e for e in events if e['typeId'] in ON_BALL_TYPE_IDS]
    pos_in_on_ball = {id(e): i for i, e in enumerate(on_ball)}

    rows = []
    for e in events:
        if e['typeId'] != PASS_TYPE_ID:
            continue
        quals = qmap(e)
        if Q['CROSS'] not in quals:
            continue
        if SET_PIECE_QUALIFIERS & set(quals.keys()):
            continue  # exclude corner / free-kick / throw-in crosses -> open play only

        end_x = quals.get(Q['PASS_END_X'])
        end_y = quals.get(Q['PASS_END_Y'])
        length = quals.get(Q['LENGTH'])
        angle = quals.get(Q['ANGLE'])
        end_x = float(end_x) if end_x is not None else np.nan
        end_y = float(end_y) if end_y is not None else np.nan
        length = float(length) if length is not None else np.nan
        angle = float(angle) if angle is not None else np.nan

        body_part = 'other'
        if Q['HEADER'] in quals: body_part = 'header'
        elif Q['RIGHT_FOOT'] in quals: body_part = 'right_foot'
        elif Q['LEFT_FOOT'] in quals: body_part = 'left_foot'

        phase = 'fast_break' if Q['FAST_BREAK'] in quals else 'regular'
        pull_back = 1 if Q['PULL_BACK'] in quals else 0

        linked_shot = shot_by_related.get(e['eventId'])
        shot_created = 1 if linked_shot is not None else 0
        goal_created = 1 if (linked_shot is not None and linked_shot['typeId'] == GOAL_TYPE_ID) else 0
        shot_x = linked_shot.get('x') if linked_shot is not None else np.nan
        shot_y = linked_shot.get('y') if linked_shot is not None else np.nan

        cleared = 0
        is_headed_clearance = 0
        clr_start_x = clr_start_y = clr_landing_x = clr_landing_y = np.nan
        clr_period = clr_elapsed = np.nan
        clr_direction = None
        clr_contestant_id = clr_player_id = clr_player_name = None
        pib = pos_in_on_ball.get(id(e))
        if pib is not None and pib + 1 < len(on_ball):
            nxt = on_ball[pib + 1]
            if nxt['typeId'] in DEFENSIVE_STOP_TYPE_IDS and nxt['contestantId'] != e['contestantId']:
                cleared = 1
                if nxt['typeId'] == CLEARANCE_TYPE_ID:
                    nquals = qmap(nxt)
                    if Q['HEADER'] in nquals:
                        nlx, nly = nquals.get(Q['PASS_END_X']), nquals.get(Q['PASS_END_Y'])
                        if nlx is not None and nly is not None:
                            is_headed_clearance = 1
                            clr_start_x, clr_start_y = nxt.get('x'), nxt.get('y')
                            clr_landing_x, clr_landing_y = float(nlx), float(nly)
                            clr_period = nxt['periodId']
                            clr_elapsed = {1: 0, 2: 45 * 60}.get(nxt['periodId'], 0) + nxt['timeMin'] * 60 + nxt['timeSec']
                            clr_direction = nquals.get(Q['DIRECTION']) or 'Unknown'
                            clr_contestant_id = nxt['contestantId']
                            clr_player_id = str(nxt.get('playerId') or 'Unknown')
                            clr_player_name = nxt.get('playerName') or 'Unknown'

        rows.append(dict(
            match=match_name, home_team=home_team, away_team=away_team,
            event_id=e['eventId'], minute=e['timeMin'], second=e['timeSec'], period=e['periodId'],
            contestant_id=e['contestantId'], player=e.get('playerName'),
            x=e.get('x'), y=e.get('y'), end_x=end_x, end_y=end_y, length=length, angle=angle,
            body_part=body_part, phase=phase, pull_back=pull_back, outcome=e.get('outcome', 0),
            shot_created=shot_created, goal_created=goal_created, cleared=cleared,
            shot_x=shot_x, shot_y=shot_y,
            is_headed_clearance=is_headed_clearance,
            clr_start_x=clr_start_x, clr_start_y=clr_start_y,
            clr_landing_x=clr_landing_x, clr_landing_y=clr_landing_y,
            clr_period=clr_period, clr_elapsed=clr_elapsed, clr_direction=clr_direction,
            clr_contestant_id=clr_contestant_id, clr_player_id=clr_player_id, clr_player_name=clr_player_name,
        ))
    return rows


files = sorted(SOURCE_DIR.glob('*.json'))
print(f'Parsing {len(files)} Premier League match files ...')
all_rows = []
for f in files:
    all_rows.extend(extract_crosses(f))

df = pd.DataFrame(all_rows)
print(f'Extracted {len(df):,} open-play (non-set-piece) crosses.')

# ----------------------------------------------------------------------------
# Step 2: feature engineering + targets
# ----------------------------------------------------------------------------
df['dist_to_goal_start'] = np.sqrt((100 - df['x']) ** 2 + (50 - df['y']) ** 2)
df['dist_to_goal_end'] = np.sqrt((100 - df['end_x']) ** 2 + (50 - df['end_y']) ** 2)
df['byline_proximity'] = 100 - df['x']
df['delivery_depth'] = df['end_x'] - df['x']
df['lateral_shift'] = (df['end_y'] - df['y']).abs()
df['end_zone_y_dist_center'] = (df['end_y'] - 50).abs()
df['wide_channel'] = np.where(df['y'] < 50, 'left', 'right')

df = df.dropna(subset=NUM_FEATURES).reset_index(drop=True)
print(f'{len(df):,} crosses retained after dropping rows with missing coordinates.')

shot_dist = np.sqrt((100 - df['shot_x']) ** 2 + (50 - df['shot_y']) ** 2)
df['danger_value'] = np.where(df['shot_created'] == 1, np.exp(-shot_dist / 12.0), 0.0)
df['danger_value'] = df['danger_value'].fillna(0.0)


def _outcome_class(r):
    if r['goal_created'] == 1: return 'goal'
    if r['shot_created'] == 1: return 'shot_no_goal'
    if r['outcome'] == 1: return 'complete_no_shot'
    return 'incomplete'


df['outcome_class'] = df.apply(_outcome_class, axis=1)

# team name resolver: for each contestantId, intersect the {home_team, away_team}
# sets of every fixture it appears in -> converges on the one constant team name.
contestant_matches = {}
for _, r in df[['contestant_id', 'home_team', 'away_team']].drop_duplicates().iterrows():
    contestant_matches.setdefault(r['contestant_id'], []).append({r['home_team'], r['away_team']})
team_name_map = {}
for cid, sets_list in contestant_matches.items():
    inter = set.intersection(*sets_list) if sets_list else set()
    team_name_map[cid] = list(inter)[0] if len(inter) == 1 else f'Unresolved({cid[:6]})'
df['team'] = df['contestant_id'].map(team_name_map)
df['opponent_team'] = np.where(df['team'] == df['home_team'], df['away_team'], df['home_team'])

X_ALL = df[FEATURE_COLS]
GROUPS = df['match']


def make_preprocessor():
    return ColumnTransformer([
        ('num', 'passthrough', NUM_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES),
    ])


# ----------------------------------------------------------------------------
# Step 3: model definitions -- 6 distinct trained models sharing one feature set
# ----------------------------------------------------------------------------
MODEL_SPECS = {
    'cross_completion': dict(
        kind='binary', target='outcome',
        desc='P(cross is retained by a teammate)'),
    'cross_chance_creation': dict(
        kind='binary', target='shot_created',
        desc='P(cross directly leads to a shot)'),
    'cross_goal_contribution': dict(
        kind='binary', target='goal_created',
        desc='P(cross directly leads to a goal) -- "xCrossGoal"'),
    'cross_defended': dict(
        kind='binary', target='cleared',
        desc='P(defending team immediately clears/intercepts the cross)'),
    'cross_delivery_value': dict(
        kind='regression', target='danger_value',
        desc='Continuous danger value of the cross (0 if no shot resulted, else proximity-to-goal decay of the resulting shot)'),
    'cross_outcome_multiclass': dict(
        kind='multiclass', target='outcome_class',
        desc='Multiclass outcome: incomplete / complete_no_shot / shot_no_goal / goal'),
}

results = {}


def get_feature_names(fitted_preprocessor):
    try:
        return list(fitted_preprocessor.get_feature_names_out())
    except Exception:
        return None


for name, spec in MODEL_SPECS.items():
    y_raw = df[spec['target']]
    gkf = GroupKFold(n_splits=N_SPLITS)

    if spec['kind'] == 'binary':
        y = y_raw.values.astype(int)
        metrics = {'auc': [], 'log_loss': [], 'brier': []}
        oof_pred = np.full(len(y), np.nan)
        fold_id = np.full(len(y), -1)
        for fold_i, (tr_idx, te_idx) in enumerate(gkf.split(X_ALL, y, GROUPS)):
            Xtr, Xte = X_ALL.iloc[tr_idx], X_ALL.iloc[te_idx]
            ytr, yte = y[tr_idx], y[te_idx]
            pos, neg = ytr.sum(), len(ytr) - ytr.sum()
            spw = max(neg / max(pos, 1), 1.0)
            pipe = Pipeline([
                ('pre', make_preprocessor()),
                ('clf', XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                                       subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                                       eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0)),
            ])
            pipe.fit(Xtr, ytr)
            p = pipe.predict_proba(Xte)[:, 1]
            oof_pred[te_idx] = p
            fold_id[te_idx] = fold_i
            metrics['auc'].append(roc_auc_score(yte, p))
            metrics['log_loss'].append(log_loss(yte, p, labels=[0, 1]))
            metrics['brier'].append(brier_score_loss(yte, p))
        cv_summary = {k: float(np.mean(v)) for k, v in metrics.items()}
        df[f'oof_{name}'] = oof_pred
        df[f'fold_{name}'] = fold_id

        pos, neg = y.sum(), len(y) - y.sum()
        spw = max(neg / max(pos, 1), 1.0)
        final_pipe = Pipeline([
            ('pre', make_preprocessor()),
            ('clf', XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                                   eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0)),
        ])
        final_pipe.fit(X_ALL, y)
        df[f'pred_{name}'] = final_pipe.predict_proba(X_ALL)[:, 1]
        feat_names = get_feature_names(final_pipe.named_steps['pre'])
        feat_imp = dict(zip(feat_names, final_pipe.named_steps['clf'].feature_importances_)) if feat_names else {}

    elif spec['kind'] == 'regression':
        y = y_raw.values.astype(float)
        metrics = {'mae': [], 'r2': []}
        oof_pred = np.full(len(y), np.nan)
        for tr_idx, te_idx in gkf.split(X_ALL, y, GROUPS):
            Xtr, Xte = X_ALL.iloc[tr_idx], X_ALL.iloc[te_idx]
            ytr, yte = y[tr_idx], y[te_idx]
            pipe = Pipeline([
                ('pre', make_preprocessor()),
                ('reg', XGBRegressor(n_estimators=250, max_depth=4, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      random_state=RANDOM_STATE, verbosity=0)),
            ])
            pipe.fit(Xtr, ytr)
            p = pipe.predict(Xte)
            oof_pred[te_idx] = p
            metrics['mae'].append(mean_absolute_error(yte, p))
            metrics['r2'].append(r2_score(yte, p))
        cv_summary = {k: float(np.mean(v)) for k, v in metrics.items()}
        df[f'oof_{name}'] = oof_pred

        final_pipe = Pipeline([
            ('pre', make_preprocessor()),
            ('reg', XGBRegressor(n_estimators=250, max_depth=4, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=RANDOM_STATE, verbosity=0)),
        ])
        final_pipe.fit(X_ALL, y)
        df[f'pred_{name}'] = final_pipe.predict(X_ALL)
        feat_names = get_feature_names(final_pipe.named_steps['pre'])
        feat_imp = dict(zip(feat_names, final_pipe.named_steps['reg'].feature_importances_)) if feat_names else {}

    else:  # multiclass
        y_labels, uniques = pd.factorize(y_raw.values)
        metrics = {'accuracy': [], 'macro_f1': []}
        oof_pred_code = np.full(len(y_labels), -1)
        for tr_idx, te_idx in gkf.split(X_ALL, y_labels, GROUPS):
            Xtr, Xte = X_ALL.iloc[tr_idx], X_ALL.iloc[te_idx]
            ytr, yte = y_labels[tr_idx], y_labels[te_idx]
            pipe = Pipeline([
                ('pre', make_preprocessor()),
                ('clf', XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                                       subsample=0.8, colsample_bytree=0.8,
                                       eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0)),
            ])
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            oof_pred_code[te_idx] = pred
            metrics['accuracy'].append(accuracy_score(yte, pred))
            metrics['macro_f1'].append(f1_score(yte, pred, average='macro'))
        cv_summary = {k: float(np.mean(v)) for k, v in metrics.items()}
        df[f'oof_{name}'] = np.where(oof_pred_code >= 0, np.array(uniques)[np.clip(oof_pred_code, 0, None)], None)

        final_pipe = Pipeline([
            ('pre', make_preprocessor()),
            ('clf', XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8,
                                   eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0)),
        ])
        final_pipe.fit(X_ALL, y_labels)
        pred_codes = final_pipe.predict(X_ALL)
        df[f'pred_{name}'] = uniques[pred_codes]
        cv_summary['classes'] = list(uniques)
        feat_names = get_feature_names(final_pipe.named_steps['pre'])
        feat_imp = dict(zip(feat_names, final_pipe.named_steps['clf'].feature_importances_)) if feat_names else {}

    joblib.dump(final_pipe, MODEL_DIR / f'model_{name}.pkl')
    results[name] = dict(desc=spec['desc'], target=spec['target'], kind=spec['kind'], cv=cv_summary,
                          cv_folds=metrics, feature_importance=feat_imp,
                          n=int(len(df)), positive_rate=float(y_raw.mean()) if spec['kind'] != 'multiclass' else None)
    print(f"[{name}] {spec['desc']}")
    print('   cv:', cv_summary)

joblib.dump(dict(feature_cols=FEATURE_COLS, num_features=NUM_FEATURES, cat_features=CAT_FEATURES,
                  qualifier_mapping=Q, results=results, team_name_map=team_name_map),
            MODEL_DIR / 'model_meta.pkl')
print('\nSaved 6 trained models to', MODEL_DIR)

# ----------------------------------------------------------------------------
# Step 4: score every cross + build player / team leaderboards
# ----------------------------------------------------------------------------
df['pred_cross_delivery_value_clipped'] = df['pred_cross_delivery_value'].clip(lower=0)

player_gb = df.groupby(['player', 'team'])
player_tbl = player_gb.agg(
    crosses=('outcome', 'size'),
    completed=('outcome', 'sum'),
    expected_completed=('pred_cross_completion', 'sum'),
    chances_created=('shot_created', 'sum'),
    expected_chances_created=('pred_cross_chance_creation', 'sum'),
    goals_created=('goal_created', 'sum'),
    expected_goals_created=('pred_cross_goal_contribution', 'sum'),
    cross_threat=('pred_cross_delivery_value_clipped', 'sum'),
).reset_index()
player_tbl = player_tbl[player_tbl['crosses'] >= MIN_CROSSES_FOR_LEADERBOARD].copy()
player_tbl['completion_rate'] = player_tbl['completed'] / player_tbl['crosses']
player_tbl['expected_completion_rate'] = player_tbl['expected_completed'] / player_tbl['crosses']
player_tbl['completion_added_value'] = player_tbl['completed'] - player_tbl['expected_completed']
player_tbl['chance_creation_rate'] = player_tbl['chances_created'] / player_tbl['crosses']
player_tbl['chance_creation_added_value'] = player_tbl['chances_created'] - player_tbl['expected_chances_created']
player_tbl['cross_threat_per_cross'] = player_tbl['cross_threat'] / player_tbl['crosses']
player_tbl = player_tbl.sort_values('cross_threat', ascending=False).reset_index(drop=True)

team_att = df.groupby('team').agg(
    crosses=('outcome', 'size'),
    completed=('outcome', 'sum'),
    expected_completed=('pred_cross_completion', 'sum'),
    chances_created=('shot_created', 'sum'),
    goals_created=('goal_created', 'sum'),
    expected_goals_created=('pred_cross_goal_contribution', 'sum'),
    cross_threat=('pred_cross_delivery_value_clipped', 'sum'),
).reset_index()
team_att['completion_rate'] = team_att['completed'] / team_att['crosses']
team_att = team_att.sort_values('cross_threat', ascending=False).reset_index(drop=True)

team_def = df.groupby('opponent_team').agg(
    crosses_faced=('cleared', 'size'),
    cleared=('cleared', 'sum'),
    expected_cleared=('pred_cross_defended', 'sum'),
    goals_conceded_from_cross=('goal_created', 'sum'),
).reset_index().rename(columns={'opponent_team': 'team'})
team_def['clearance_rate'] = team_def['cleared'] / team_def['crosses_faced']
team_def['expected_clearance_rate'] = team_def['expected_cleared'] / team_def['crosses_faced']
team_def['clearance_added_value'] = team_def['cleared'] - team_def['expected_cleared']
team_def = team_def.sort_values('clearance_added_value', ascending=False).reset_index(drop=True)

model_summary = pd.DataFrame([
    dict(model=name, description=spec['desc'], target=spec['target'], type=spec['kind'],
         n_crosses=results[name]['n'],
         positive_rate=results[name]['positive_rate'],
         **{f'cv_{k}': v for k, v in results[name]['cv'].items() if k != 'classes'})
    for name, spec in MODEL_SPECS.items()
])

event_cols = ['match', 'team', 'opponent_team', 'player', 'period', 'minute', 'second',
              'x', 'y', 'end_x', 'end_y', 'length', 'angle', 'body_part', 'phase', 'pull_back',
              'outcome', 'shot_created', 'goal_created', 'cleared'] + \
             [c for c in df.columns if c.startswith('pred_') and not c.endswith('clipped')] + \
             [c for c in df.columns if c.startswith('oof_')]

events_out = df[event_cols].copy()
events_out.to_csv(OUTPUT_DIR / 'open_play_cross_events_scored.csv', index=False)

report_path = OUTPUT_DIR / 'premier_league_cross_models_report.xlsx'
with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
    model_summary.to_excel(writer, sheet_name='Model Summary', index=False)
    player_tbl.to_excel(writer, sheet_name='Player Leaderboard', index=False)
    team_att.to_excel(writer, sheet_name='Team Attacking', index=False)
    team_def.to_excel(writer, sheet_name='Team Defensive', index=False)

print(f'\nSaved event-level scored crosses -> {OUTPUT_DIR / "open_play_cross_events_scored.csv"}')
print(f'Saved report workbook          -> {report_path}')
print(f'  Sheets: Model Summary, Player Leaderboard ({len(player_tbl)} players, '
      f'min {MIN_CROSSES_FOR_LEADERBOARD} crosses), Team Attacking, Team Defensive')

print('\n=== MODEL SUMMARY ===')
print(model_summary.to_string(index=False))
print('\n=== TOP 10 PLAYERS BY CROSS THREAT ===')
print(player_tbl[['player', 'team', 'crosses', 'completion_rate', 'chance_creation_added_value',
                   'expected_goals_created', 'cross_threat']].head(10).to_string(index=False))


# ----------------------------------------------------------------------------
# Step 5: visuals (Waltzing Analytics dark brand style)
# ----------------------------------------------------------------------------
VIZ_DIR = OUTPUT_DIR / 'visuals'
CROSS_MAP_DIR = VIZ_DIR / 'cross_maps'
VIZ_DIR.mkdir(parents=True, exist_ok=True)
CROSS_MAP_DIR.mkdir(parents=True, exist_ok=True)

BINARY_MODELS = [n for n, s in MODEL_SPECS.items() if s['kind'] == 'binary']
BINARY_COLORS = [C_NAVY, C_INDIGO, C_AMBER, C_CORAL]
BINARY_LABELS = {
    'cross_completion': 'Completion', 'cross_chance_creation': 'Chance creation',
    'cross_goal_contribution': 'Goal contribution', 'cross_defended': 'Defended/cleared',
}

# --- Flagship: per-team "Crosses That Created a Shot" pitch maps -----------
def make_cross_map(team_name, sub_df, out_path):
    n_total = len(sub_df)
    n_goals = int(sub_df['goal_created'].sum())

    fig = plt.figure(figsize=(11, 10.6))
    fig.patch.set_facecolor(BG)

    fig.text(0.5, 0.955, team_name, fontsize=26, fontweight='bold', ha='center', color='white')
    fig.text(0.5, 0.918, 'Crosses That Created a Shot  ·  Open Play Only  ·  Premier League  ·  2025-26',
             fontsize=12, ha='center', color=TXT_SUB)

    pitch_ax = fig.add_axes([0.04, 0.10, 0.92, 0.72])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                           linewidth=1.1, half=True)
    pitch.draw(ax=pitch_ax)

    for _, e in sub_df.sort_values('goal_created').iterrows():
        is_goal = e['goal_created'] == 1
        color = C_GREEN if is_goal else C_AMBER
        lw = 2.2 if is_goal else 1.3
        alpha = 0.9 if is_goal else 0.55
        pitch.lines(e['x'], e['y'], e['shot_x'], e['shot_y'], ax=pitch_ax, color=color,
                    lw=lw, alpha=alpha, zorder=2, comet=True)
        pitch.scatter(e['shot_x'], e['shot_y'], ax=pitch_ax, s=40 if is_goal else 22,
                      color=color, edgecolors='white', linewidths=0.8, alpha=alpha, zorder=3)

    legend_elems = [
        Line2D([0], [0], color=C_GREEN, linewidth=2.5, label=f'Goal ({n_goals})'),
        Line2D([0], [0], color=C_AMBER, linewidth=2.5, label=f'Shot, no goal ({n_total - n_goals})'),
    ]
    fig.legend(handles=legend_elems, loc='lower center', bbox_to_anchor=(0.5, 0.885),
               ncol=2, frameon=False, fontsize=10, labelcolor=TXT_CAP)

    conv = n_goals / n_total if n_total else 0
    caption = f'{n_total} open-play crosses led to a shot this season  ·  {n_goals} goals  ·  {conv:.0%} conversion'
    fig.text(0.5, 0.045, caption, fontsize=11.5, ha='center', color=TXT_CAP)

    top = sub_df['player'].value_counts().head(5)
    if len(top):
        top_str = '  ·  '.join(f'{name} ({n})' for name, n in top.items())
        fig.text(0.5, 0.022, f'Top providers: {top_str}', fontsize=9, ha='center', color=TXT_MUTE)

    add_footer(fig, FOOTER_NOTE)
    add_logo(fig)
    fig.savefig(out_path, dpi=200, facecolor=BG)
    plt.close(fig)


shot_df = df[df['shot_created'] == 1]
for team_name in sorted(df['team'].unique()):
    safe_name = team_name.replace(' ', '_').replace('&', 'and')
    make_cross_map(team_name, shot_df[shot_df['team'] == team_name], CROSS_MAP_DIR / f'{safe_name}.png')
print(f'Saved {df["team"].nunique()} per-team cross maps -> {CROSS_MAP_DIR}')

# --- 3) Cross origin volume + delivery-value heatmaps -----------------------
fig = plt.figure(figsize=(15, 8.6), facecolor=BG)
fig.text(0.5, 0.955, 'Where the Danger Comes From', fontsize=22, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.912, 'Open-play cross origin volume vs. average predicted delivery value  ·  Premier League  ·  2025-26',
         fontsize=12, ha='center', color=TXT_SUB)

ax1 = fig.add_axes([0.03, 0.10, 0.44, 0.76])
pitch1 = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE, linewidth=1.1, half=False)
pitch1.draw(ax=ax1)
bs = pitch1.bin_statistic(df['x'], df['y'], statistic='count', bins=(24, 16))
bs['statistic'] = np.where(bs['statistic'] == 0, np.nan, bs['statistic'])
pitch1.heatmap(bs, ax=ax1, cmap='YlOrBr', edgecolors=None, alpha=0.92)
ax1.set_title('Origin volume (all crosses)', color=TXT_CAP, fontsize=12, y=1.02)

ax2 = fig.add_axes([0.51, 0.10, 0.44, 0.76])
pitch2 = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE, linewidth=1.1, half=False)
pitch2.draw(ax=ax2)
bs2 = pitch2.bin_statistic(df['x'], df['y'], values=df['pred_cross_delivery_value_clipped'],
                            statistic='mean', bins=(24, 16))
counts2 = pitch2.bin_statistic(df['x'], df['y'], statistic='count', bins=(24, 16))
bs2['statistic'] = np.where(counts2['statistic'] < 3, np.nan, bs2['statistic'])
pitch2.heatmap(bs2, ax=ax2, cmap='YlOrBr', edgecolors=None, alpha=0.92)
ax2.set_title('Avg. predicted delivery value (min 3 crosses/cell)', color=TXT_CAP, fontsize=12, y=1.02)

add_footer(fig, FOOTER_NOTE)
add_logo(fig, width=0.11)
fig.savefig(VIZ_DIR / '03_cross_origin_heatmaps.png', dpi=200, facecolor=BG, bbox_inches='tight')
plt.close(fig)

# --- 4) Top 15 crossers ------------------------------------------------------
top15 = player_tbl.head(15).iloc[::-1]
fig = plt.figure(figsize=(11, 9.5), facecolor=BG)
ax = fig.add_axes([0.27, 0.08, 0.65, 0.74])
style_axes_dark(ax)
ylabels = top15['player'] + '  (' + top15['team'].str.replace(' FC', '', regex=False) + ')'
bars = ax.barh(ylabels, top15['cross_threat'], color=C_NAVY, height=0.6)
for b, v in zip(bars, top15['cross_threat']):
    ax.text(v + 0.05, b.get_y() + b.get_height() / 2, f'{v:.1f}', va='center', color='white', fontsize=9)
ax.set_xlabel('Cross threat (sum of predicted delivery value)')
ax.tick_params(axis='y', colors=TXT_CAP, labelsize=10)

fig.text(0.5, 0.94, 'Top Crossers', fontsize=22, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.90, f'By cross threat  ·  min {MIN_CROSSES_FOR_LEADERBOARD} open-play crosses  ·  Premier League  ·  2025-26',
         fontsize=12, ha='center', color=TXT_SUB)
add_footer(fig, FOOTER_NOTE)
add_logo(fig, width=0.14)
fig.savefig(VIZ_DIR / '04_top_crossers.png', dpi=200, facecolor=BG, bbox_inches='tight')
plt.close(fig)

# --- 5) Team attacking vs defensive quadrant scatter ------------------------
team_plot = team_att.merge(team_def[['team', 'clearance_added_value']], on='team')
fig = plt.figure(figsize=(11, 9.5), facecolor=BG)
ax = fig.add_axes([0.10, 0.08, 0.87, 0.74])
style_axes_dark(ax)
x_mid = team_plot['cross_threat'].mean()
ax.axvline(x_mid, color=TXT_MUTE, lw=1, ls='--')
ax.axhline(0, color=TXT_MUTE, lw=1, ls='--')
colors = np.where(team_plot['clearance_added_value'] >= 0, C_GREEN, C_CORAL)
ax.scatter(team_plot['cross_threat'], team_plot['clearance_added_value'], s=80, c=colors,
           edgecolors='white', linewidths=0.5, zorder=3)
texts = [ax.text(r['cross_threat'], r['clearance_added_value'], r['team'].replace(' FC', ''),
                  fontsize=8.5, color=TXT_CAP)
         for _, r in team_plot.iterrows()]
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color=TXT_MUTE, lw=0.6))
except ImportError:
    pass
ax.set_xlabel('Attacking cross threat (season total)')
ax.set_ylabel('Defensive clearance added value\n(crosses cleared − expected)')

fig.text(0.5, 0.94, 'Team Cross Profile', fontsize=22, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.90, 'Creating danger vs. defending it  ·  Premier League  ·  2025-26', fontsize=12, ha='center', color=TXT_SUB)
add_footer(fig, FOOTER_NOTE)
add_logo(fig, width=0.13)
fig.savefig(VIZ_DIR / '05_team_attack_vs_defend.png', dpi=200, facecolor=BG, bbox_inches='tight')
plt.close(fig)

# --- 6) Delivery type breakdown ----------------------------------------------
_bp_full_order = ['right_foot', 'left_foot', 'header', 'other']
_bp_full_colors = {'right_foot': C_NAVY, 'left_foot': C_INDIGO, 'header': C_AMBER, 'other': C_GREEN}
bp_counts = df['body_part'].value_counts()
bp_order = [b for b in _bp_full_order if bp_counts.get(b, 0) > 0]
bp_colors = [_bp_full_colors[b] for b in bp_order]
bp_stats = df.groupby('body_part').agg(
    crosses=('outcome', 'size'), completion_rate=('outcome', 'mean'),
    chance_creation_rate=('shot_created', 'mean'),
).reindex(bp_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), facecolor=BG)
fig.subplots_adjust(top=0.72, bottom=0.14, wspace=0.28)

ax = axes[0]
style_axes_dark(ax)
bars = ax.bar(bp_order, bp_stats['crosses'], color=bp_colors, width=0.55)
for b, v in zip(bars, bp_stats['crosses']):
    ax.text(b.get_x() + b.get_width() / 2, v + 30, f'{int(v):,}', ha='center', color='white', fontsize=9.5)
ax.set_title('Crosses by delivery type', color='white', fontsize=12, loc='left')

ax = axes[1]
style_axes_dark(ax)
width = 0.35
xpos = np.arange(len(bp_order))
ax.bar(xpos - width / 2, bp_stats['completion_rate'], width, color=C_NAVY, label='Completion rate')
ax.bar(xpos + width / 2, bp_stats['chance_creation_rate'], width, color=C_AMBER, label='Chance-creation rate')
ax.set_xticks(xpos)
ax.set_xticklabels(bp_order)
ax.set_title('Outcome rate by delivery type', color='white', fontsize=12, loc='left')
ax.legend(frameon=False, fontsize=9.5, labelcolor=TXT_CAP)

fig.text(0.5, 0.94, 'Delivery Type Breakdown', fontsize=22, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.885, 'Open-play crosses  ·  Premier League  ·  2025-26', fontsize=12, ha='center', color=TXT_SUB)
add_footer(fig, FOOTER_NOTE)
add_logo(fig, width=0.12)
fig.savefig(VIZ_DIR / '06_delivery_type_breakdown.png', dpi=200, facecolor=BG, bbox_inches='tight')
plt.close(fig)

print(f'\nSaved WA-brand pitch/summary visuals -> {VIZ_DIR}')
for p in sorted(VIZ_DIR.glob('*.png')):
    print('  -', p.name)


# ----------------------------------------------------------------------------
# Step 6: headed-clearance landing model, chained onto the cross pipeline
# ----------------------------------------------------------------------------
# Same methodology as Ecuador 2026/Scripts/header_clearance_landing_model.py:
# extract every headed clearance (typeId 12 + qualifier 15 "header") from the
# same Premier League match files, train a MultiOutputRegressor(GradientBoosting)
# to predict its landing x/y, then apply it to the subset of our open-play
# crosses that were cleared by a header (is_headed_clearance == 1) so every
# such cross also gets a predicted clearance-landing zone.
LANDING_DIR = SOURCE_DIR / 'ClearanceLandingModel'
LANDING_DIR.mkdir(parents=True, exist_ok=True)


def clearance_zone_x(x):
    if x < 33.333: return 'defensive_third'
    if x < 66.667: return 'middle_third'
    return 'attacking_third'


def clearance_zone_y(y):
    if y < 33.333: return 'left_channel'
    if y < 66.667: return 'central_channel'
    return 'right_channel'


def extract_all_headed_clearances(path: Path, team_map):
    match_name = path.stem
    _, events = parse_match_file(path)
    rows = []
    for e in events:
        if e['typeId'] != CLEARANCE_TYPE_ID:
            continue
        quals = qmap(e)
        if Q['HEADER'] not in quals:
            continue
        lx, ly = quals.get(Q['PASS_END_X']), quals.get(Q['PASS_END_Y'])
        sx, sy = e.get('x'), e.get('y')
        if lx is None or ly is None or sx is None or sy is None:
            continue
        lx, ly = float(lx), float(ly)
        elapsed = {1: 0, 2: 45 * 60}.get(e['periodId'], 0) + e['timeMin'] * 60 + e['timeSec']
        rows.append(dict(
            match_id=match_name, team=team_map.get(e['contestantId'], e['contestantId']),
            player_id=str(e.get('playerId') or 'Unknown'), player_name=e.get('playerName') or 'Unknown',
            period_id=e['periodId'], elapsed_seconds=elapsed,
            start_x=sx, start_y=sy, start_x_zone=clearance_zone_x(sx), start_y_zone=clearance_zone_y(sy),
            direction=quals.get(Q['DIRECTION']) or 'Unknown',
            distance_from_own_goal=sx, distance_from_center=abs(sy - 50),
            landing_x=lx, landing_y=ly,
        ))
    return rows


clr_rows = []
for f in files:
    clr_rows.extend(extract_all_headed_clearances(f, team_name_map))
clr_df = pd.DataFrame(clr_rows)
print(f'\nExtracted {len(clr_df):,} headed clearances (all, not just from crosses) for the landing model.')
clr_df.to_csv(LANDING_DIR / 'headed_clearances_dataset.csv', index=False)

LANDING_NUM_FEATURES = ['period_id', 'elapsed_seconds', 'start_x', 'start_y',
                         'distance_from_own_goal', 'distance_from_center']
LANDING_CAT_FEATURES = ['team', 'player_id', 'start_x_zone', 'start_y_zone', 'direction']
LANDING_FEATURE_COLS = LANDING_NUM_FEATURES + LANDING_CAT_FEATURES
LANDING_TARGET_COLS = ['landing_x', 'landing_y']

landing_preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), LANDING_NUM_FEATURES),
    ('categorical', OneHotEncoder(handle_unknown='ignore', min_frequency=5), LANDING_CAT_FEATURES),
])
landing_model = Pipeline([
    ('preprocess', landing_preprocessor),
    ('model', MultiOutputRegressor(GradientBoostingRegressor(
        n_estimators=220, learning_rate=0.04, max_depth=3, min_samples_leaf=8, random_state=RANDOM_STATE))),
])

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
clr_train_idx, clr_test_idx = next(splitter.split(clr_df, groups=clr_df['match_id']))
clr_train, clr_test = clr_df.iloc[clr_train_idx], clr_df.iloc[clr_test_idx]
landing_model.fit(clr_train[LANDING_FEATURE_COLS], clr_train[LANDING_TARGET_COLS])


def evaluate_landing(y_true, y_pred):
    m = {}
    for i, axis in enumerate(['x', 'y']):
        truth, pred = y_true.iloc[:, i].to_numpy(), y_pred[:, i]
        m[f'mae_{axis}'] = float(mean_absolute_error(truth, pred))
        m[f'rmse_{axis}'] = float(mean_squared_error(truth, pred) ** 0.5)
        m[f'r2_{axis}'] = float(r2_score(truth, pred))
    err = np.sqrt(((y_true.to_numpy() - y_pred) ** 2).sum(axis=1))
    m['mean_landing_error'] = float(err.mean())
    m['median_landing_error'] = float(np.median(err))
    m['pct_within_10_pitch_units'] = float((err <= 10).mean())
    m['pct_within_15_pitch_units'] = float((err <= 15).mean())
    return m


clr_test_pred = landing_model.predict(clr_test[LANDING_FEATURE_COLS])
landing_metrics = dict(
    n_headed_clearances=int(len(clr_df)), n_matches=int(clr_df['match_id'].nunique()),
    n_train=int(len(clr_train)), n_test=int(len(clr_test)),
    test=evaluate_landing(clr_test[LANDING_TARGET_COLS], clr_test_pred),
)
joblib.dump(dict(model=landing_model, feature_cols=LANDING_FEATURE_COLS, target_cols=LANDING_TARGET_COLS,
                  metrics=landing_metrics), LANDING_DIR / 'headed_clearance_landing_model.joblib')
with open(LANDING_DIR / 'model_metrics.json', 'w') as fobj:
    json.dump(landing_metrics, fobj, indent=2)
print(f"Landing model test mean error: {landing_metrics['test']['mean_landing_error']:.2f} pitch units "
      f"(n_train={landing_metrics['n_train']:,}, n_test={landing_metrics['n_test']:,})")

# --- Chain onto the cross dataset -------------------------------------------
headed_mask = df['is_headed_clearance'] == 1
chain_df = df.loc[headed_mask, ['contestant_id']].copy()
chain_df['team'] = df.loc[headed_mask, 'clr_contestant_id'].map(team_name_map)
chain_df['player_id'] = df.loc[headed_mask, 'clr_player_id']
chain_df['period_id'] = df.loc[headed_mask, 'clr_period']
chain_df['elapsed_seconds'] = df.loc[headed_mask, 'clr_elapsed']
chain_df['start_x'] = df.loc[headed_mask, 'clr_start_x']
chain_df['start_y'] = df.loc[headed_mask, 'clr_start_y']
chain_df['start_x_zone'] = chain_df['start_x'].apply(clearance_zone_x)
chain_df['start_y_zone'] = chain_df['start_y'].apply(clearance_zone_y)
chain_df['direction'] = df.loc[headed_mask, 'clr_direction']
chain_df['distance_from_own_goal'] = chain_df['start_x']
chain_df['distance_from_center'] = (chain_df['start_y'] - 50).abs()

pred = landing_model.predict(chain_df[LANDING_FEATURE_COLS])
df.loc[headed_mask, 'pred_clr_landing_x'] = np.clip(pred[:, 0], 0, 100)
df.loc[headed_mask, 'pred_clr_landing_y'] = np.clip(pred[:, 1], 0, 100)
df.loc[headed_mask, 'clr_landing_error'] = np.sqrt(
    (df.loc[headed_mask, 'clr_landing_x'] - df.loc[headed_mask, 'pred_clr_landing_x']) ** 2
    + (df.loc[headed_mask, 'clr_landing_y'] - df.loc[headed_mask, 'pred_clr_landing_y']) ** 2
)
# clearance distance gained: how far up the pitch the header travels (own-goal direction = 0 -> 100 = opp goal)
df.loc[headed_mask, 'clr_distance_gained'] = df.loc[headed_mask, 'clr_landing_x'] - df.loc[headed_mask, 'clr_start_x']

n_headed = int(headed_mask.sum())
print(f'\nChained landing model onto {n_headed:,} of the {int(df["cleared"].sum()):,} cleared crosses '
      f'({n_headed / max(int(df["cleared"].sum()), 1):.0%} were headed clearances with usable landing data).')

# team clearance-quality leaderboard: who clears crosses furthest / most reliably beyond expectation
clr_team_tbl = df.loc[headed_mask].assign(
    defending_team=df.loc[headed_mask, 'clr_contestant_id'].map(team_name_map)
).groupby('defending_team').agg(
    headed_clearances=('clr_distance_gained', 'size'),
    avg_distance_gained=('clr_distance_gained', 'mean'),
    avg_landing_error=('clr_landing_error', 'mean'),
).reset_index().sort_values('avg_distance_gained', ascending=False)

with pd.ExcelWriter(report_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    clr_team_tbl.to_excel(writer, sheet_name='Clearance Landing (Team)', index=False)
print(f'Added "Clearance Landing (Team)" sheet to {report_path}')

# --- Visual: where headed clearances from crosses actually land -------------
fig = plt.figure(figsize=(11, 10.6), facecolor=BG)
fig.text(0.5, 0.955, 'Where Headed Clearances Go', fontsize=26, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.918, 'Crosses Cleared by a Header  ·  Actual vs. Predicted Landing  ·  Premier League  ·  2025-26',
         fontsize=12, ha='center', color=TXT_SUB)

pitch_ax = fig.add_axes([0.04, 0.10, 0.92, 0.72])
pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE, linewidth=1.1, half=False)
pitch.draw(ax=pitch_ax)

clr_plot = df.loc[headed_mask].sample(min(600, n_headed), random_state=RANDOM_STATE)
pitch.scatter(clr_plot['clr_landing_x'], clr_plot['clr_landing_y'], ax=pitch_ax, s=20,
              color=C_AMBER, alpha=0.5, edgecolors='none', zorder=2, label='Actual landing')
pitch.scatter(clr_plot['pred_clr_landing_x'], clr_plot['pred_clr_landing_y'], ax=pitch_ax, s=20,
              color=C_INDIGO, alpha=0.5, edgecolors='none', zorder=3, label='Predicted landing')

legend_elems = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=C_AMBER, markersize=8, label='Actual landing'),
    Line2D([0], [0], marker='o', color='none', markerfacecolor=C_INDIGO, markersize=8, label='Predicted landing'),
]
fig.legend(handles=legend_elems, loc='lower center', bbox_to_anchor=(0.5, 0.885),
           ncol=2, frameon=False, fontsize=10, labelcolor=TXT_CAP)

caption = (f'{n_headed:,} crosses cleared by a header this season  ·  '
           f'landing model mean error {landing_metrics["test"]["mean_landing_error"]:.1f} pitch units '
           f'(n={landing_metrics["n_test"]:,} held-out test clearances)')
fig.text(0.5, 0.045, caption, fontsize=11.5, ha='center', color=TXT_CAP)

top_clr = clr_team_tbl.head(5)
top_str = '  ·  '.join(f"{r['defending_team'].replace(' FC', '')} (+{r['avg_distance_gained']:.1f})"
                        for _, r in top_clr.iterrows())
fig.text(0.5, 0.022, f'Furthest-clearing defenses: {top_str}', fontsize=9, ha='center', color=TXT_MUTE)

add_footer(fig, FOOTER_NOTE)
add_logo(fig)
fig.savefig(VIZ_DIR / '07_clearance_landing.png', dpi=200, facecolor=BG)
plt.close(fig)
print(f'Saved -> {VIZ_DIR / "07_clearance_landing.png"}')


# ----------------------------------------------------------------------------
# Step 7: "rest attack" positioning -- combine cross data with clearance
# landing data to show where to station players for the second ball.
# ----------------------------------------------------------------------------
REST_ATTACK_DIR = VIZ_DIR / 'rest_attack_maps'
REST_ATTACK_DIR.mkdir(parents=True, exist_ok=True)

# re-save the scored CSV now that Step 6 has added the clearance-landing columns
event_cols_full = event_cols + [
    'is_headed_clearance', 'clr_start_x', 'clr_start_y', 'clr_landing_x', 'clr_landing_y',
    'pred_clr_landing_x', 'pred_clr_landing_y', 'clr_landing_error', 'clr_distance_gained', 'wide_channel',
]
df[event_cols_full].to_csv(OUTPUT_DIR / 'open_play_cross_events_scored.csv', index=False)


def top_zone_centers(bs, n=3):
    flat = np.nan_to_num(bs['statistic'].flatten(), nan=-1.0)
    n = min(n, int((flat > 0).sum()))
    order = np.argsort(flat)[::-1][:n]
    rows, cols = np.unravel_index(order, bs['statistic'].shape)
    return list(zip(bs['cx'][rows, cols], bs['cy'][rows, cols], flat[order]))


headed = df.loc[df['is_headed_clearance'] == 1].copy()

# --- 8) League-wide second-ball recovery zones ------------------------------
fig = plt.figure(figsize=(11, 10.6), facecolor=BG)
fig.text(0.5, 0.955, 'Second-Ball Recovery Zones', fontsize=26, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.918, 'Where Headed Clearances From Your Crosses Land  ·  Premier League  ·  2025-26',
         fontsize=12, ha='center', color=TXT_SUB)

pitch_ax = fig.add_axes([0.04, 0.10, 0.92, 0.72])
pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE, linewidth=1.1, half=False)
pitch.draw(ax=pitch_ax)

bs = pitch.bin_statistic(headed['clr_landing_x'], headed['clr_landing_y'], statistic='count', bins=(18, 12))
bs['statistic'] = np.where(bs['statistic'] == 0, np.nan, bs['statistic'])
pitch.heatmap(bs, ax=pitch_ax, cmap='YlOrBr', edgecolors=None, alpha=0.85, zorder=1)

for rank, (cx, cy, cnt) in enumerate(top_zone_centers(bs, n=3), start=1):
    pitch_ax.scatter([cx], [cy], s=260, marker='*', color='white', edgecolors=C_INDIGO,
                      linewidths=1.5, zorder=4)
    pitch_ax.annotate(f'{rank}', (cx, cy), color=BG, fontsize=9, fontweight='bold',
                       ha='center', va='center', zorder=5)

legend_elems = [Line2D([0], [0], marker='*', color='none', markerfacecolor='white',
                        markeredgecolor=C_INDIGO, markersize=14, label='Top recovery zone (rank)')]
fig.legend(handles=legend_elems, loc='lower center', bbox_to_anchor=(0.5, 0.885),
           ncol=1, frameon=False, fontsize=10, labelcolor=TXT_CAP)

caption = f'{len(headed):,} headed clearances from open-play crosses this season  ·  density = where the loose ball ends up'
fig.text(0.5, 0.045, caption, fontsize=11.5, ha='center', color=TXT_CAP)
fig.text(0.5, 0.022, 'Position rest-attack players in the starred zones to win the second ball',
         fontsize=9, ha='center', color=TXT_MUTE)

add_footer(fig, FOOTER_NOTE)
add_logo(fig)
fig.savefig(VIZ_DIR / '08_second_ball_recovery_zones.png', dpi=200, facecolor=BG)
plt.close(fig)
print(f'Saved -> {VIZ_DIR / "08_second_ball_recovery_zones.png"}')

# --- 9) Recovery zones split by which flank the cross came from -------------
fig = plt.figure(figsize=(15, 8.6), facecolor=BG)
fig.text(0.5, 0.955, 'Second-Ball Recovery Zones by Cross Side', fontsize=22, fontweight='bold', ha='center', color='white')
fig.text(0.5, 0.912, 'Where headed clearances land, split by which flank the cross was delivered from  ·  Premier League  ·  2025-26',
         fontsize=12, ha='center', color=TXT_SUB)

for i, side in enumerate(['left', 'right']):
    ax = fig.add_axes([0.03 + i * 0.48, 0.10, 0.44, 0.76])
    p = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE, linewidth=1.1, half=False)
    p.draw(ax=ax)
    sub = headed[headed['wide_channel'] == side]
    bs_side = p.bin_statistic(sub['clr_landing_x'], sub['clr_landing_y'], statistic='count', bins=(18, 12))
    bs_side['statistic'] = np.where(bs_side['statistic'] == 0, np.nan, bs_side['statistic'])
    p.heatmap(bs_side, ax=ax, cmap='YlOrBr', edgecolors=None, alpha=0.85)
    for rank, (cx, cy, cnt) in enumerate(top_zone_centers(bs_side, n=2), start=1):
        ax.scatter([cx], [cy], s=220, marker='*', color='white', edgecolors=C_INDIGO, linewidths=1.5, zorder=4)
        ax.annotate(f'{rank}', (cx, cy), color=BG, fontsize=8.5, fontweight='bold', ha='center', va='center', zorder=5)
    ax.set_title(f'Crosses from the {side} side  (n={len(sub):,})', color=TXT_CAP, fontsize=12, y=1.02)

add_footer(fig, FOOTER_NOTE)
add_logo(fig, width=0.11)
fig.savefig(VIZ_DIR / '09_second_ball_recovery_by_side.png', dpi=200, facecolor=BG)
plt.close(fig)
print(f'Saved -> {VIZ_DIR / "09_second_ball_recovery_by_side.png"}')

# --- Per-team "rest attack" positioning maps ---------------------------------
def make_rest_attack_map(team_name, sub, out_path):
    n = len(sub)
    fig = plt.figure(figsize=(11, 10.6), facecolor=BG)
    fig.text(0.5, 0.955, team_name, fontsize=26, fontweight='bold', ha='center', color='white')
    fig.text(0.5, 0.918, 'Rest-Attack Positioning  ·  Where Your Crosses Get Headed Clear  ·  2025-26',
             fontsize=12, ha='center', color=TXT_SUB)

    # full pitch, not half -- headed clearances land in the defensive third,
    # which sits outside a half=True (attacking-half-only) crop
    pitch_ax = fig.add_axes([0.04, 0.10, 0.92, 0.72])
    p = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE, linewidth=1.1, half=False)
    p.draw(ax=pitch_ax)

    if n >= 5:
        bs_t = p.bin_statistic(sub['clr_landing_x'], sub['clr_landing_y'], statistic='count', bins=(12, 8))
        bs_t['statistic'] = np.where(bs_t['statistic'] == 0, np.nan, bs_t['statistic'])
        p.heatmap(bs_t, ax=pitch_ax, cmap='YlOrBr', edgecolors=None, alpha=0.8, zorder=1)
        for rank, (cx, cy, cnt) in enumerate(top_zone_centers(bs_t, n=2), start=1):
            p.scatter([cx], [cy], ax=pitch_ax, s=240, marker='*', color='white', edgecolors=C_INDIGO,
                      linewidths=1.5, zorder=4)
            p.annotate(f'{rank}', (cx, cy), ax=pitch_ax, color=BG, fontsize=9, fontweight='bold',
                       ha='center', va='center', zorder=5)
    else:
        p.scatter(sub['clr_landing_x'], sub['clr_landing_y'], ax=pitch_ax, s=40, color=C_AMBER,
                   alpha=0.7, edgecolors='white', linewidths=0.6, zorder=3)

    legend_elems = [Line2D([0], [0], marker='*', color='none', markerfacecolor='white',
                            markeredgecolor=C_INDIGO, markersize=14, label='Recommended rest-attack zone')]
    fig.legend(handles=legend_elems, loc='lower center', bbox_to_anchor=(0.5, 0.885),
               ncol=1, frameon=False, fontsize=10, labelcolor=TXT_CAP)

    avg_gain = sub['clr_distance_gained'].mean() if n else float('nan')
    caption = f'{n} of your open-play crosses were headed clear this season  ·  avg. clearance travels {avg_gain:.1f} pitch units'
    fig.text(0.5, 0.045, caption, fontsize=11.5, ha='center', color=TXT_CAP)
    fig.text(0.5, 0.022, 'Station midfielders/full-backs in the starred zones to win the second ball off your own crosses',
             fontsize=9, ha='center', color=TXT_MUTE)

    add_footer(fig, FOOTER_NOTE)
    add_logo(fig)
    fig.savefig(out_path, dpi=200, facecolor=BG)
    plt.close(fig)


for team_name in sorted(df['team'].unique()):
    safe_name = team_name.replace(' ', '_').replace('&', 'and')
    make_rest_attack_map(team_name, headed[headed['team'] == team_name], REST_ATTACK_DIR / f'{safe_name}.png')
print(f'Saved {df["team"].nunique()} per-team rest-attack maps -> {REST_ATTACK_DIR}')


# ----------------------------------------------------------------------------
# Step 8: academic-style model diagnostics (classic scientific-paper look)
# ----------------------------------------------------------------------------
# Separate from the WA dark brand pitch maps above: white background, black
# ink, Okabe-Ito colorblind-safe palette, serif type, gridlines, figure
# numbers/captions and explicit methodology notes -- for methods-section use
# rather than media/presentation use.
ACA_BG = '#ffffff'
ACA_INK = '#000000'
ACA_MUTED = '#595959'
ACA_GRID = '#d9d9d9'
OK = {  # Okabe & Ito (2008) colorblind-safe palette
    'black': '#000000', 'orange': '#E69F00', 'sky': '#56B4E9', 'green': '#009E73',
    'yellow': '#F0E442', 'blue': '#0072B2', 'vermillion': '#D55E00', 'purple': '#CC79A7',
}
plt.rcParams.update({
    'figure.facecolor': ACA_BG, 'axes.facecolor': ACA_BG,
    'axes.edgecolor': ACA_INK, 'axes.labelcolor': ACA_INK,
    'text.color': ACA_INK, 'xtick.color': ACA_INK, 'ytick.color': ACA_INK,
    'font.family': 'serif', 'font.size': 10.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': ACA_GRID, 'grid.linewidth': 0.6,
    'legend.frameon': True, 'legend.edgecolor': ACA_INK, 'legend.framealpha': 1.0,
})

ACA_MODEL_COLORS = [OK['blue'], OK['orange'], OK['green'], OK['vermillion']]
N_MATCHES = df['match'].nunique()
METHOD_NOTE = (f'5-fold GroupKFold cross-validation, grouped by match to prevent leakage. '
               f'n = {len(df):,} open-play crosses, {N_MATCHES} Premier League matches (2025-26).')


def add_caption(fig, fig_num, title, note, y_title=0.965, y_note=0.02):
    fig.text(0.5, y_title, f'Figure {fig_num}. {title}', fontsize=13, fontweight='bold', ha='center', color=ACA_INK)
    fig.text(0.5, y_note, note, fontsize=8.5, ha='center', color=ACA_MUTED, wrap=True)


# --- Figure 1: model performance summary with fold-level error bars --------
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.subplots_adjust(top=0.78, bottom=0.22, wspace=0.35)

ax = axes[0]
aucs = [results[n]['cv']['auc'] for n in BINARY_MODELS]
auc_std = [np.std(results[n]['cv_folds']['auc']) for n in BINARY_MODELS]
labels = [BINARY_LABELS[n] for n in BINARY_MODELS]
ypos = np.arange(len(labels))
ax.barh(ypos, aucs, xerr=auc_std, color=ACA_MODEL_COLORS, height=0.55,
        edgecolor=ACA_INK, linewidth=0.7, capsize=4, error_kw=dict(ecolor=ACA_INK, elinewidth=1))
ax.set_yticks(ypos, labels)
ax.set_xlim(0.5, 0.9)
ax.axvline(0.5, color=ACA_MUTED, lw=1, ls='--')
for y, v, s in zip(ypos, aucs, auc_std):
    ax.text(v + s + 0.01, y, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('AUC (mean ± 1 SD across folds)')
ax.set_title('(a) Classifier discrimination', fontsize=11, loc='left')
ax.invert_yaxis()
ax.grid(axis='y', visible=False)

ax = axes[1]
reg = results['cross_delivery_value']
bp = ax.boxplot([reg['cv_folds']['r2'], reg['cv_folds']['mae']], tick_labels=['R²', 'MAE'],
                patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], [OK['blue'], OK['orange']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
    patch.set_edgecolor(ACA_INK)
for element in ['whiskers', 'caps', 'medians']:
    plt.setp(bp[element], color=ACA_INK)
ax.set_title('(b) Delivery-value regression', fontsize=11, loc='left')
ax.set_ylabel('Score (per fold)')

ax = axes[2]
mc = results['cross_outcome_multiclass']
bp = ax.boxplot([mc['cv_folds']['accuracy'], mc['cv_folds']['macro_f1']],
                tick_labels=['Accuracy', 'Macro F1'], patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], [OK['blue'], OK['orange']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
    patch.set_edgecolor(ACA_INK)
for element in ['whiskers', 'caps', 'medians']:
    plt.setp(bp[element], color=ACA_INK)
ax.set_ylim(0, 1)
ax.set_title('(c) 4-class outcome model', fontsize=11, loc='left')

add_caption(fig, 1, 'Cross-validated performance of the six trained models',
            METHOD_NOTE + ' Boxplots in (b)/(c) show the distribution of the fold-level scores.')
fig.savefig(VIZ_DIR / '01_model_performance_summary.png', dpi=220, bbox_inches='tight', facecolor=ACA_BG)
plt.close(fig)

# --- Figure 2: ROC curves with cross-validated variability band ------------
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
mean_fpr = np.linspace(0, 1, 100)
for ax, name, color in zip(axes.flat, BINARY_MODELS, ACA_MODEL_COLORS):
    tprs = []
    fold_aucs = results[name]['cv_folds']['auc']
    for fold_i in sorted(df[f'fold_{name}'].unique()):
        if fold_i < 0:
            continue
        mask = df[f'fold_{name}'] == fold_i
        yte = df.loc[mask, MODEL_SPECS[name]['target']].astype(int)
        p = df.loc[mask, f'oof_{name}']
        fpr, tpr, _ = roc_curve(yte, p)
        ax.plot(fpr, tpr, color=color, lw=0.8, alpha=0.35)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
    tprs = np.array(tprs)
    mean_tpr, std_tpr = tprs.mean(axis=0), tprs.std(axis=0)
    ax.plot(mean_fpr, mean_tpr, color=color, lw=2.2, label='Mean ROC')
    ax.fill_between(mean_fpr, np.clip(mean_tpr - std_tpr, 0, 1), np.clip(mean_tpr + std_tpr, 0, 1),
                     color=color, alpha=0.18, label='± 1 SD')
    ax.plot([0, 1], [0, 1], color=ACA_MUTED, lw=1, ls='--')
    ax.set_title(f'{BINARY_LABELS[name]}  (AUC = {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f})',
                 fontsize=10.5, loc='left')
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.legend(loc='lower right', fontsize=8)

add_caption(fig, 2, 'Cross-validated ROC curves by fold', METHOD_NOTE +
            ' Thin lines show individual folds; bold line is the fold-mean with ±1 SD band.', y_title=0.975, y_note=0.005)
fig.tight_layout(rect=(0, 0.02, 1, 0.955))
fig.savefig(VIZ_DIR / '02_roc_curves.png', dpi=220, bbox_inches='tight', facecolor=ACA_BG)
plt.close(fig)

# --- Figure 3: calibration (reliability) curves -----------------------------
fig = plt.figure(figsize=(11, 9.5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.32, top=0.86, bottom=0.10, left=0.09, right=0.97)
ax_cal = fig.add_subplot(gs[0])
ax_hist = fig.add_subplot(gs[1])
ax_cal.plot([0, 1], [0, 1], color=ACA_MUTED, lw=1, ls='--', label='Perfectly calibrated')
for name, color in zip(BINARY_MODELS, ACA_MODEL_COLORS):
    mask = df[f'oof_{name}'].notna()
    yte = df.loc[mask, MODEL_SPECS[name]['target']].astype(int)
    p = df.loc[mask, f'oof_{name}']
    frac_pos, mean_pred = calibration_curve(yte, p, n_bins=10, strategy='quantile')
    ax_cal.plot(mean_pred, frac_pos, marker='o', ms=5, lw=1.6, color=color, label=BINARY_LABELS[name])
    ax_hist.hist(p, bins=30, range=(0, 1), histtype='step', color=color, lw=1.4, density=True)
ax_cal.set_xlabel('Mean predicted probability (per decile bin)')
ax_cal.set_ylabel('Observed frequency')
ax_cal.set_title('(a) Reliability diagram (deciles, out-of-fold predictions)', fontsize=11, loc='left')
ax_cal.legend(loc='upper left', fontsize=9)
ax_hist.set_xlabel('Predicted probability')
ax_hist.set_ylabel('Density')
ax_hist.set_title('(b) Distribution of predicted probabilities', fontsize=11, loc='left')
add_caption(fig, 3, 'Calibration of the four binary classifiers', METHOD_NOTE +
            ' Points on the diagonal in (a) indicate well-calibrated probabilities. Completion and chance-creation '
            'are visibly over-confident, a known side effect of the scale_pos_weight class-imbalance correction '
            'used at training time -- it improves ranking (AUC) at the cost of calibrated probabilities.')
fig.savefig(VIZ_DIR / '10_calibration_curves.png', dpi=220, bbox_inches='tight', facecolor=ACA_BG)
plt.close(fig)

# --- Figure 4: feature importance (small multiples) -------------------------
IMP_ORDER = ['cross_completion', 'cross_chance_creation', 'cross_goal_contribution',
             'cross_defended', 'cross_delivery_value', 'cross_outcome_multiclass']
IMP_TITLES = {
    'cross_completion': 'Completion', 'cross_chance_creation': 'Chance creation',
    'cross_goal_contribution': 'Goal contribution', 'cross_defended': 'Defended/cleared',
    'cross_delivery_value': 'Delivery value (regression)', 'cross_outcome_multiclass': 'Outcome (4-class)',
}


def clean_feat_name(n):
    return n.replace('num__', '').replace('cat__', '')


fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, name in zip(axes.flat, IMP_ORDER):
    imp = results[name]['feature_importance']
    items = sorted(imp.items(), key=lambda kv: kv[1], reverse=True)[:8][::-1]
    labels = [clean_feat_name(k) for k, v in items]
    vals = [v for k, v in items]
    ax.barh(labels, vals, color=OK['blue'], edgecolor=ACA_INK, linewidth=0.5, height=0.6)
    ax.set_title(IMP_TITLES[name], fontsize=10.5, loc='left')
    ax.set_xlabel('Importance (gain)')
    ax.tick_params(axis='y', labelsize=8.5)
add_caption(fig, 4, 'Top-8 feature importances per model', METHOD_NOTE +
            ' Importances are XGBoost gain-based scores from the model refit on all data; feature names are '
            'post-one-hot-encoding.', y_title=0.975, y_note=0.005)
fig.tight_layout(rect=(0, 0.03, 1, 0.94))
fig.savefig(VIZ_DIR / '11_feature_importance.png', dpi=220, bbox_inches='tight', facecolor=ACA_BG)
plt.close(fig)

# --- Figure 5: confusion matrix (multiclass, out-of-fold) ------------------
mc_classes = results['cross_outcome_multiclass']['cv']['classes']
mask = df['oof_cross_outcome_multiclass'].notna()
cm = confusion_matrix(df.loc[mask, 'outcome_class'], df.loc[mask, 'oof_cross_outcome_multiclass'],
                       labels=mc_classes, normalize='true')

fig, ax = plt.subplots(figsize=(7.5, 6.8))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(mc_classes)), mc_classes, rotation=30, ha='right')
ax.set_yticks(range(len(mc_classes)), mc_classes)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
for i in range(len(mc_classes)):
    for j in range(len(mc_classes)):
        ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center',
                 color='white' if cm[i, j] > 0.5 else ACA_INK, fontsize=10)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('Row-normalized frequency (recall)')
ax.set_title('Confusion matrix, out-of-fold predictions', fontsize=11, loc='left')
add_caption(fig, 5, 'Confusion matrix for the 4-class cross-outcome model', METHOD_NOTE +
            ' The model rarely predicts the rare classes (shot_no_goal, goal) even when they occur -- the '
            'headline 79.7% accuracy is driven almost entirely by the majority "incomplete" class; macro F1 '
            '(0.343) is the more honest summary of performance on this task.', y_title=0.98, y_note=-0.05)
fig.savefig(VIZ_DIR / '12_confusion_matrix.png', dpi=220, bbox_inches='tight', facecolor=ACA_BG)
plt.close(fig)

# --- Figure 6: residual diagnostics (delivery-value regression) ------------
mask = df['oof_cross_delivery_value'].notna()
actual = df.loc[mask, 'danger_value']
pred = df.loc[mask, 'oof_cross_delivery_value']
resid = actual - pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
ax = axes[0]
hb = ax.hexbin(pred, actual, gridsize=35, cmap='Blues', mincnt=1, bins='log')
lims = [min(pred.min(), actual.min()), max(pred.max(), actual.max())]
ax.plot(lims, lims, color=OK['vermillion'], lw=1.5, ls='--', label='y = x')
ax.set_xlabel('Predicted delivery value')
ax.set_ylabel('Actual delivery value')
ax.set_title('(a) Predicted vs. actual (out-of-fold)', fontsize=11, loc='left')
ax.legend(loc='upper left', fontsize=9)
cb = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('log10(count)')

ax = axes[1]
ax.hist(resid, bins=50, color=OK['blue'], edgecolor=ACA_INK, linewidth=0.4, alpha=0.85)
ax.axvline(0, color=OK['vermillion'], lw=1.5, ls='--')
ax.axvline(resid.mean(), color=ACA_INK, lw=1.2, ls=':', label=f'Mean = {resid.mean():.4f}')
ax.set_xlabel('Residual (actual − predicted)')
ax.set_ylabel('Count')
ax.set_title(f'(b) Residual distribution (SD = {resid.std():.4f})', fontsize=11, loc='left')
ax.legend(loc='upper right', fontsize=9)
add_caption(fig, 6, 'Residual diagnostics for the cross-delivery-value regression', METHOD_NOTE +
            ' The secondary mode of positive residuals near +0.3 shows the model under-predicting the crosses '
            'that actually produced a high-quality chance -- consistent with the low R² and the shrink-to-zero '
            'behaviour typical of regressing on a mostly-zero target.', y_title=0.99, y_note=0.005)
fig.tight_layout(rect=(0, 0.05, 1, 0.90))
fig.savefig(VIZ_DIR / '13_residual_diagnostics.png', dpi=220, bbox_inches='tight', facecolor=ACA_BG)
plt.close(fig)

print(f'\nSaved 6 academic-style diagnostic figures -> {VIZ_DIR}')
print('  - 01_model_performance_summary.png (reworked, academic style)')
print('  - 02_roc_curves.png (reworked, academic style)')
print('  - 10_calibration_curves.png')
print('  - 11_feature_importance.png')
print('  - 12_confusion_matrix.png')
print('  - 13_residual_diagnostics.png')